# Load Datasets

### Imports y carga de datos

In [ ]:
import pandas as pd
import numpy as np

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)


A partir de ahora, solo voy a trabajar con el v_02 y construir a partir de eso.

In [ ]:
match_vector = pd.read_csv(
    "../data/processed/match_vector_v02.csv"
)

match_vector.head()

# Baseline Evaluation

### Reutilización de funciones útiles

#### TRAIN - TEST by Year

In [ ]:
def split_train_test_by_year(df, target_col="target"):
    train_df = df[df["year"] == 2018].copy()
    test_df = df[df["year"] == 2022].copy()

    columns_to_drop = [
        "match_id",
        "match_date",
        "kick_off",
        "year",
        "home_team",
        "away_team",
        "home_score",
        "away_score",
        "competition_stage",
        target_col
    ]

    X_train = train_df.drop(columns=columns_to_drop)
    y_train = train_df[target_col]

    X_test = test_df.drop(columns=columns_to_drop)
    y_test = test_df[target_col]

    return X_train, X_test, y_train, y_test

    

#### Evaluación de Modelos

In [ ]:
def evaluate_model(model, X_train, y_train, X_test, y_test, model_name=None, dataset_name=None):
    """
    Entrena y evalúa un modelo de predicción.
    Devuelve dict con resultados.
    """
    
    if model_name is None:                       
        model_name = model.__class__.__name__


    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)

    cm = confusion_matrix(y_test, predictions)

    report = classification_report(
        y_test,
        predictions,
        output_dict=True,
        zero_division = 0
    )

    return {
        "dataset": dataset_name,
        "model_name": model_name,
        "model": model,
        "predictions": predictions,
        "accuracy": accuracy,
        "confusion_matrix": cm,
        "classification_report": report,
    }



#### Feature Importance

In [ ]:
def get_feature_importance(result, feature_names, top_n=20):
    """
    Creates a feature importance DataFrame for models with feature_importances_.
    """

    model = result["model"]

    if not hasattr(model, "feature_importances_"):
        raise ValueError(
            f"{result['model_name']} does not have feature_importances_."
        )

    if len(model.feature_importances_) != len(feature_names):
        raise ValueError(
            f"Length mismatch: model has {len(model.feature_importances_)} importances "
            f"but {len(feature_names)} feature names were provided."
        )

    importance_df = pd.DataFrame({
        "feature": feature_names,
        "importance": model.feature_importances_
    })

    importance_df = importance_df.sort_values(
        by="importance",
        ascending=False
    ).reset_index(drop=True)

    return importance_df.head(top_n)




# Feature Importance Analysis

# Low-importance Feature Removal

# Correlation Analysis

# Redundant Feature Removal

# Retrain Models

# Compare v02 - v03

# Save match_Vector_03.csv

# Conclusions